# EDA Feature Engineering - 10 Meaningful Features

This notebook is a controlled experiment. It keeps the same data source, split strategy, preprocessing style, and model configurations as the baseline model comparison. The only intended change is adding 10 EDA-based features, then retraining Random Forest and SVM.

In [ ]:
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name == "feature_engineering" else Path.cwd()
DATA_PATH = ROOT_DIR / "data" / "diabetic_data_clean_common.csv"
BASELINE_RANKING_PATH = ROOT_DIR / "train" / "model_comparison" / "outputs" / "practical_model_ranking.csv"
OUTPUT_DIR = ROOT_DIR / "train" / "feature_engineering" / "eda_10_features_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "readmitted_binary"
RANDOM_STATE = 42

DATA_PATH, BASELINE_RANKING_PATH, OUTPUT_DIR

## 1. Load And Split Data

Use the same source data and split strategy as the baseline notebook. Split first to avoid leakage. Thresholds such as Q3 are fitted only from the train set.

In [ ]:
df = pd.read_csv(DATA_PATH)

y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Train target ratio:")
print(y_train.value_counts(normalize=True).rename("ratio"))

## 2. Create 10 EDA-Based Features

The added features are intentionally limited and explainable:

1. `total_prior_visits`
2. `has_prior_inpatient`
3. `meds_per_day`
4. `labs_per_day`
5. `is_long_stay`
6. `is_senior`
7. `a1c_abnormal`
8. `insulin_changed`
9. `num_diabetes_drugs_used`
10. `num_unique_diag_groups`

In [ ]:
DRUG_COLUMNS = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
]


def safe_divide(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan).fillna(0)


def fit_fe_params(X_train):
    return {
        "q3_time_in_hospital": X_train["time_in_hospital"].quantile(0.75),
    }


def add_10_eda_features(X, params):
    X_fe = X.copy()

    X_fe["total_prior_visits"] = (
        X_fe["number_outpatient"] + X_fe["number_emergency"] + X_fe["number_inpatient"]
    )
    X_fe["has_prior_inpatient"] = (X_fe["number_inpatient"] > 0).astype(int)
    X_fe["meds_per_day"] = safe_divide(X_fe["num_medications"], X_fe["time_in_hospital"])
    X_fe["labs_per_day"] = safe_divide(X_fe["num_lab_procedures"], X_fe["time_in_hospital"])
    X_fe["is_long_stay"] = (X_fe["time_in_hospital"] >= params["q3_time_in_hospital"]).astype(int)
    X_fe["is_senior"] = (X_fe["age_ordinal"] >= 6).astype(int)
    X_fe["a1c_abnormal"] = X_fe["A1Cresult"].fillna("None").isin([">7", ">8"]).astype(int)
    X_fe["insulin_changed"] = X_fe["insulin"].isin(["Up", "Down"]).astype(int)
    X_fe["num_diabetes_drugs_used"] = X_fe[DRUG_COLUMNS].fillna("No").ne("No").sum(axis=1)
    X_fe["num_unique_diag_groups"] = X_fe[["diag_1_group", "diag_2_group", "diag_3_group"]].fillna("Unknown").nunique(axis=1)

    return X_fe


fe_params = fit_fe_params(X_train)

X_train_fe = add_10_eda_features(X_train, fe_params)
X_val_fe = add_10_eda_features(X_val, fe_params)
X_test_fe = add_10_eda_features(X_test, fe_params)

new_features = sorted(set(X_train_fe.columns) - set(X_train.columns))
print("New features:", len(new_features))
pd.DataFrame({"new_feature": new_features})

## 3. Preprocess

Remove raw target text and raw ICD diagnosis codes, then scale numeric features and one-hot encode categorical features.

In [ ]:
DROP_COLUMNS = ["readmitted", "diag_1", "diag_2", "diag_3", "age", "age_midpoint"]

X_train_model = X_train_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_train_fe.columns])
X_val_model = X_val_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_val_fe.columns])
X_test_model = X_test_fe.drop(columns=[col for col in DROP_COLUMNS if col in X_test_fe.columns])

categorical_cols = X_train_model.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X_train_model.columns if col not in categorical_cols]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Input columns before encoding:", X_train_model.shape[1])

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("onehot", make_one_hot_encoder())]), categorical_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_model)
X_val_processed = preprocessor.transform(X_val_model)
X_test_processed = preprocessor.transform(X_test_model)
feature_names = preprocessor.get_feature_names_out()

print("Processed X_train:", X_train_processed.shape)
print("Processed X_val:", X_val_processed.shape)
print("Processed X_test:", X_test_processed.shape)

## 4. Retrain Random Forest And SVM

Use the same model configurations as baseline model comparison. This keeps the comparison fair: baseline vs baseline + 10 EDA features.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "SVM": LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE,
    ),
}


def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate_model(name, model):
    start_time = time.time()
    model.fit(X_train_processed, y_train)
    train_time = time.time() - start_time

    y_pred = model.predict(X_val_processed)
    y_score = get_score_for_auc(model, X_val_processed)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1_score": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, y_score) if y_score is not None else None,
        "train_time_sec": train_time,
    }
    cm = pd.DataFrame(
        confusion_matrix(y_val, y_pred),
        index=["actual_0", "actual_1"],
        columns=["predicted_0", "predicted_1"],
    )
    report = pd.DataFrame(classification_report(y_val, y_pred, output_dict=True, zero_division=0)).T
    return metrics, cm, report, model


results = []
confusion_matrices = {}
classification_reports = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    metrics, cm, report, trained_model = evaluate_model(name, model)
    results.append(metrics)
    confusion_matrices[name] = cm
    classification_reports[name] = report
    trained_models[name] = trained_model
    print(
        f"Done {name}: F1={metrics['f1_score']:.4f}, "
        f"Recall={metrics['recall']:.4f}, Precision={metrics['precision']:.4f}, "
        f"Accuracy={metrics['accuracy']:.4f}, ROC-AUC={metrics['roc_auc']:.4f}"
    )

results_df = pd.DataFrame(results).sort_values(["f1_score", "roc_auc", "accuracy"], ascending=False).reset_index(drop=True)
results_df

## 5. Fair Comparison With Baseline

Compare the same models before and after adding the 10 EDA features.

In [ ]:
baseline_df = pd.read_csv(BASELINE_RANKING_PATH)
baseline_df = baseline_df[baseline_df["model"].isin(["Random Forest", "SVM"])].copy()
baseline_df["version"] = "Baseline"

feature_df = results_df.copy()
feature_df["version"] = "+10 EDA features"

comparison_cols = ["model", "version", "accuracy", "precision", "recall", "f1_score", "roc_auc"]
fair_comparison_df = pd.concat(
    [baseline_df[comparison_cols], feature_df[comparison_cols]],
    ignore_index=True,
).sort_values(["model", "version"]).reset_index(drop=True)

delta_rows = []
for model_name in ["Random Forest", "SVM"]:
    baseline_row = baseline_df[baseline_df["model"] == model_name].iloc[0]
    feature_row = feature_df[feature_df["model"] == model_name].iloc[0]
    delta_rows.append({
        "model": model_name,
        "delta_accuracy": feature_row["accuracy"] - baseline_row["accuracy"],
        "delta_precision": feature_row["precision"] - baseline_row["precision"],
        "delta_recall": feature_row["recall"] - baseline_row["recall"],
        "delta_f1_score": feature_row["f1_score"] - baseline_row["f1_score"],
        "delta_roc_auc": feature_row["roc_auc"] - baseline_row["roc_auc"],
    })

delta_df = pd.DataFrame(delta_rows)

print("Baseline vs +10 EDA features:")
display(fair_comparison_df)

print("Metric deltas (+10 EDA features - Baseline):")
display(delta_df)

## 6. Save Outputs

In [ ]:
best_model_name = results_df.loc[0, "model"]

pd.DataFrame({"new_feature": new_features}).to_csv(OUTPUT_DIR / "new_10_features.csv", index=False)
pd.DataFrame([fe_params]).to_csv(OUTPUT_DIR / "feature_engineering_params.csv", index=False)
pd.Series(feature_names, name="feature_name").to_csv(OUTPUT_DIR / "feature_names.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
fair_comparison_df.to_csv(OUTPUT_DIR / "baseline_vs_10_eda_features.csv", index=False)
delta_df.to_csv(OUTPUT_DIR / "baseline_vs_10_eda_feature_deltas.csv", index=False)

for name, cm in confusion_matrices.items():
    safe_name = name.lower().replace(" ", "_")
    cm.to_csv(OUTPUT_DIR / f"{safe_name}_validation_confusion_matrix.csv")
    classification_reports[name].to_csv(OUTPUT_DIR / f"{safe_name}_validation_classification_report.csv")

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
joblib.dump(trained_models[best_model_name], OUTPUT_DIR / "best_validation_model.joblib")

print("Best validation model:", best_model_name)
print(f"Saved outputs to: {OUTPUT_DIR}")